# Data Reading

In [0]:
df = spark.read.format("parquet").option("inferSchema", "true").load('abfss://bronze@salesprojectmgr.dfs.core.windows.net')

In [0]:
display(df)

# Data Transformation

In [0]:
from pyspark.sql.functions import *

In [0]:
# Extract the category from the 'Model_ID' column by splitting on '-' and taking the first part

df = df.withColumn('Model_Category',split(df['Model_ID'], '-').getItem(0))
display(df)


In [0]:
# Create a new column 'Revenue_per_Unit' by dividing 'Revenue' by 'Units_Sold'
df = df.withColumn('Revenue_per_Unit',(df['Revenue'] / df['Units_Sold']))
display(df)

# AD-HOC

In [0]:
# Group by 'Year' and 'BranchName', aggregate total units sold, and sort by year ascending and total units sold descending
df_grouped = df.groupBy('Year', 'BranchName') \
    .agg(sum('Units_Sold').alias('Total_Units_Sold')) \
    .sort('Year', 'Total_Units_Sold', ascending=[True, False])

display(df_grouped)

Databricks visualization. Run in Databricks to view.

# Data Writing to Silver layer

In [0]:
df.write.format("parquet").mode("overwrite").save("abfss://silver@salesprojectmgr.dfs.core.windows.net")

# Querying Silver Data


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@salesprojectmgr.dfs.core.windows.net`